# 01 · Loading
## Opening AFM images and high-speed AFM videos in Python

In this notebook we get AFM data *into* Python and learn what it looks like once it is there. Everything downstream; levelling, filtering, measuring, exporting depends on this one idea: **once an AFM image is a NumPy array, the file format stops mattering.**

We will use the **AFMReader** loader library to open AFM images and videos from a variety of AFM manufacturers and software packages so we don't have to write file parsers.

---

**Contents**

1. Packages and imports
2. Opening images and videos
3. Basic viewing and manipulation
4. Saving NumPy arrays

## 1 · Packages and imports

Run the cell below. **If it errors, flag it now** — it's the one-stop check that your
environment is ready.

In [ ]:

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# AFMReader: one loader function per format. The `LoadFile` switchboard picks the
# right one based on the file extension, so a single call handles any format.
from AFMReader.general_loader import LoadFile

# playNano can be used to work with high-speed or video AFM data but also contains helpful colourmaps 
#that are registered with maptplotlib upon import.
import playnano 

print("Environment looks good!")


We are using **AFMReader** to open the different AFM file types. Once loaded everything is plain NumPy.

## 2 · Opening images and videos

The sample files live in `data/samples/`. These sample files are from the following public data deposits (full citations in `DATA_SOURCES.md`):

- Meinhardt et al. (2024), block copolymer crystallization — [10.5281/zenodo.14202742](https://doi.org/10.5281/zenodo.14202742)
- Mohammad et al. (2025), SRC self-association — [10.5281/zenodo.14943298](https://doi.org/10.5281/zenodo.14943298)
- Aghebat Rafat et al. (2020), AFM raw data — [10.5281/zenodo.3685805](https://doi.org/10.5281/zenodo.3685805)
- Pyne et al. (2021), DNA minicircles — [10.6084/m9.figshare.13116890.v1](https://doi.org/10.6084/m9.figshare.13116890.v1)
- Heath (2025), bacteriorhodopsin membrane imaging — [10.5281/zenodo.17105214](https://doi.org/10.5281/zenodo.17105214)
- Chau et al. (2026), DNA origami separation — [10.5518/1770](https://doi.org/10.5518/1770)

In [ ]:

DATA_DIR = Path(".") / "data" / "samples"

for p in sorted(DATA_DIR.iterdir()):
    print(p.name)


### 2.1 · Opening a single image with AFMReader

AFMReader loader returns a tuple containing the image data as a NumPy array and a scaling factor to convert pixel dimension to nanometres:

```python
image, pixel_to_nm = load_xxx(file_path, channel)
```

- `image` — a 2D NumPy array of heights (in nm).
- `pixel_to_nm` — a single `float`: the width of one pixel in nanometres. Multiply a
  pixel distance by this to get a real distance.

The `channel` says which data channel to read. For Bruker `.spm` files the height
channel is usually `"Height"` (other common ones: `"ZSensor"`); for `.ibw` it's
often `"HeightRetrace"`. The right name depends on how the file was saved. If a channel 
is not present the **AFMReader** error will let you know which channels are available. 

The quickest way is the **`LoadFile` switchboard** — it looks at the file extension
and calls the correct loader for you.

In [ ]:

# --- EDIT THIS to point at one of your image samples ---
image_path = DATA_DIR / "barcode5.ibw"

image, pixel_to_nm = LoadFile(image_path, channel="HeightRetrace").load()

print(f"Loaded:      {image_path.name}")
print(f"Type:        {type(image)}")
print(f"Shape:       {image.shape}   (height, width in pixels)")
print(f"dtype:       {image.dtype}")
print(f"Pixel size:  {pixel_to_nm:.4f} nm/pixel")


The AFM image data from the file has now been loaded as `image` as an ordinary NumPy array. You could equally call the specific loader directly if you prefer being explicit:

```python
from AFMReader.ibw import load_ibw
image, pixel_to_nm = load_ibw(image_path, channel="HeightRetrace")
```

### Exercise 2.1

Load a **different** image from `data/samples/` — ideally a different format — with
`LoadFile`. Print its shape and pixel size. Notice the loading code is identical
regardless of format; only the `channel` name may differ.

In [ ]:

# Your turn:
# image2, pixel_to_nm2 = LoadFile(DATA_DIR / "...", channel="Height").load()
# print(image2.shape, pixel_to_nm2)


### 2.2 · Opening a video with AFMReader

High-speed AFM videos are a series of AFM images stacked together. Depending on the
system used to take the video, the frames can either be **individual files** (one per
frame) or a **single file** that holds the whole movie.

**Folder of individual frames** (e.g. `.jpk` files) — load each frame as a 2-D array
and stack them into one 3-D array:

```python
from AFMReader.jpk import load_jpk

video_folder = DATA_DIR / "jpk_video_folder"

# Sort so frames end up in acquisition order, not arbitrary filesystem order.
frame_paths = sorted(video_folder.glob("*.jpk"))

frame_list = []
for path in frame_paths:
    img, pixel_to_nm = load_jpk(path, channel="height_trace")  # 2-D array per file
    frame_list.append(img)

frames = np.stack(frame_list)   # -> 3-D array (n_frames, height, width)
print(frames.shape, f"{pixel_to_nm:.4f} nm/pixel")
```

**Single-file videos** (`.h5-jpk`, `.asd`) — **AFMReader** opens these directly, just
like an image, but hands back a 3-D array:

```python
video, pixel_to_nm = LoadFile(video_path, channel="height_trace").load()
```

- `video` — 3D NumPy array `(n_frames, height, width)` of heights (in nm).
- `pixel_to_nm` — a single `float`: the pixel width in nanometres, shared by all frames.

> ⚠️ **Channel names vary by format.** `.h5-jpk` (and `.jpk` frames) use
> `"height_trace"`, but `.asd` files use different channel names (e.g. `"TP"`) — so the
> same `channel` string won't work for both. Check your file's channels if a load fails.

The specific loaders (`load_h5jpk()` and `load_asd()`) return **extra metadata** that
the `LoadFile` switchboard drops — `load_h5jpk` gives per-frame timestamps and
`load_asd` a metadata object:

```python
from AFMReader.h5_jpk import load_h5jpk
video, pixel_to_nm, timestamps = load_h5jpk(video_path, channel="height_trace")
```

See the [AFMReader docs](https://afm-spm.github.io/AFMReader/) for the full per-format
metadata.

In [ ]:

# --- EDIT THIS to point at one of your single-file video samples ---
video_path = DATA_DIR / "save-2025.07.01-11.57.12.721.h5-jpk"

video, video_pixel_to_nm = LoadFile(video_path, channel="height_trace").load()

print(f"Loaded:      {video_path.name}")
print(f"Type:        {type(video)}")
print(f"Shape:       {video.shape}   (n_frames, height, width)")
print(f"dtype:       {video.dtype}")
print(f"Pixel size:  {video_pixel_to_nm:.4f} nm/pixel")
print(f"Frames:      {video.shape[0]}")


From here on we treat `video` as a plain NumPy array. A single frame is
`video[i]`, which is a 2-D array exactly like the image from AFMReader.

## 3 · Basic viewing and manipulation

Everything below is generic NumPy + matplotlib. It works the same on the single image
and on any frame of the video, because they're both just 2D arrays.

### 3.1 · Displaying an array

`plt.imshow` draws a 2-D array. Two AFM conventions:

- **`origin="lower"`** puts the first row at the bottom, the usual AFM orientation.
- Add a **colorbar** so height has units. (`afm_brown` is registered by playNano on
  import; `"viridis"` or `"afmhot"` work too.)

In [ ]:

plt.figure(figsize=(4, 4))
plt.imshow(image, cmap="afm_brown", origin="lower")
plt.colorbar(label="Height (nm)")
plt.title("Single image")
plt.axis("off")
plt.show()

### 3.2 · Indexing into the video

A 3D array indexes by frame first (N, H, W). Pulling frames out and subtracting two of
them, is just array arithmetic.

In [ ]:
n = video.shape[0]
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, idx in zip(axes, [0, n // 2, n - 1]):
    im = ax.imshow(video[idx], cmap="afm_brown", origin="lower")
    ax.set_title(f"Frame {idx}")
    ax.axis("off")
fig.colorbar(im, ax=axes, label="Height (nm)", shrink=0.8)
plt.show()

### 3.3 · A scan line as a 1-D profile

Slice one row out of a frame to get a height profile across the surface. Convert the
x-axis from pixels to nanometres by multiplying by the pixel size.

Look at the overall **tilt / bow** — the surface isn't flat even over bare substrate.
Removing that is exactly what *levelling* (the next notebook) is for.

In [ ]:
frame0 = video[0]
row = frame0.shape[0] // 2

profile = frame0[row, :]
x_nm = np.arange(profile.size) * video_pixel_to_nm

plt.figure(figsize=(6, 3))
plt.plot(x_nm, profile)
plt.xlabel("Distance (nm)")
plt.ylabel("Height (nm)")
plt.title(f"Line profile through row {row}")
plt.tight_layout()
plt.show()

## 4 · Saving NumPy arrays

To pass data to the next notebook we save the arrays to disk with plain NumPy. Two
functions cover almost everything:

- **`np.save`** — one array → one `.npy` file.
- **`np.savez_compressed`** — several arrays (+ small metadata) → one `.npz` file.

⚠️ **Save the pixel size alongside the array.** The raw heights are useless for real
measurements without the nm-per-pixel scale, and a bare `.npy` doesn't carry it. So we
bundle the scale (and the timestamps) into the `.npz` with the array.

In [ ]:
OUT_DIR = Path(".") / "data" / "arrays"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Single image: array + its scale.
np.savez_compressed(
    OUT_DIR / "image.npz",
    image=image,
    pixel_to_nm=pixel_to_nm,
)

# Video: the 3-D array + scale.
np.savez_compressed(
    OUT_DIR / "movie.npz",
    movie=video,
    pixel_to_nm=video_pixel_to_nm,
)

print("Saved:")
print(" ", OUT_DIR / "image.npz")
print(" ", OUT_DIR / "movie.npz")

Check the files were correctly generated ready for the next notebook...

In [ ]:
image_file = np.load(OUT_DIR / "image.npz")
print("Keys:", list(image_file.keys()))

data_file = np.load(OUT_DIR / "movie.npz")
print("Keys:", list(data_file.keys()))

image_reloaded = image_file["image"]
image_scale_reloaded = float(image_file["pixel_to_nm"])


video_reloaded = data_file["movie"]
video_scale_reloaded = float(data_file["pixel_to_nm"])

assert np.allclose(image_reloaded, image)
assert np.allclose(video_reloaded[0], video[0])
print("Round-trip verified — ready for notebook 02.")